In [31]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


import time
import matplotlib.pyplot as plt

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(0.5), # data augmentation
    transforms.RandomRotation(10),      
    transforms.ToTensor(),
])

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root='datav3/training/', transform=train_transform)

print(f"Total training images: {len(full_dataset)}")
print(f"Classes found: {full_dataset.classes}")

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_size = len(full_dataset)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size

train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

Total training images: 311
Classes found: ['hand', 'no_hand']
Train: 217 | Val: 46 | Test: 48


In [33]:
class TinyCNN(torch.nn.Module):
    def __init__(self, num_outputs):
        super(TinyCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.rl1 = nn.ReLU()
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
        self.rl2 = nn.ReLU()

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.rl3 = nn.ReLU()

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        
        self.linear1 = nn.Linear(64, 32)
        self.rl4 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)
        self.linear2 = nn.Linear(32, num_outputs)

    def forward(self, x):
        x = self.rl1(self.conv1(x))
        x = self.rl2(self.conv2(x))
        x = self.rl3(self.conv3(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.rl4(self.linear1(x))
        x = self.dropout1(x)
        x = self.linear2(x)
        return x

In [39]:
def init_weights(m):
    if type(m) == torch.nn.Linear or type(m) == torch.nn.Conv2d:
        torch.nn.init.xavier_uniform_(m.weight)
        
num_classes = 2
model = TinyCNN(num_outputs=num_classes).to(device)
model.apply(init_weights)

TinyCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (rl1): ReLU()
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (rl2): ReLU()
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (rl3): ReLU()
  (pool): AdaptiveAvgPool2d(output_size=1)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=64, out_features=32, bias=True)
  (rl4): ReLU()
  (dropout1): Dropout(p=0.4, inplace=False)
  (linear2): Linear(in_features=32, out_features=2, bias=True)
)

In [40]:
def correct(logits, y):
    y_hat = logits.argmax(axis=1)
    return (y_hat == y).float().sum()
    
def evaluate_metric(model, data_iter, metric):
    c = 0.
    n = 0.
    model.eval()
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            c += metric(logits, y)
            n += len(y)
    return float(c / n)

In [ ]:
from torch.utils.data import Subset

# Use the clean transform (no rotations) for the sanity check
full_dataset = datasets.ImageFolder(root='datav3/training/', transform=transform)

subset_indices = [0, 1, total_size-1, total_size-2] 
mini_dataset = Subset(full_dataset, subset_indices)

mini_loader = DataLoader(mini_dataset, batch_size=4, shuffle=False)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
num_epochs = 400


losses = []
train_accs = []
test_accs = []

for epoch in range(num_epochs):
    model.train()
    start_time = time.perf_counter()
    train_loss = 0.0
    train_correct_count = 0
    train_total = 0
    
    for images, labels in mini_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss_val = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()
        
        losses.append(loss_val.item())
        train_loss += loss_val.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct_count += (predicted == labels).sum().item()

    train_acc = 100 * train_correct_count / train_total
    val_acc = evaluate_metric(model, mini_loader, correct) * 100
    
    train_accs.append(train_acc / 100)
    test_accs.append(val_acc / 100)
    
    end_time = time.perf_counter()
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Acc: {val_acc:.2f}% | Duration: {end_time - start_time:.3f}s")
    print("-" * 30)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Training batch')
plt.ylabel('Cross entropy loss')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Training accuracy')
plt.plot(test_accs, label='Validation accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')
plt.show()

Epoch [1/400]
Train Loss: 0.0974 | Train Acc: 75.00%
Val Acc: 50.00% | Duration: 0.086s
------------------------------
Epoch [2/400]
Train Loss: 0.0986 | Train Acc: 50.00%
Val Acc: 50.00% | Duration: 0.085s
------------------------------
Epoch [3/400]
Train Loss: 0.0991 | Train Acc: 50.00%
Val Acc: 50.00% | Duration: 0.079s
------------------------------
Epoch [4/400]
Train Loss: 0.0997 | Train Acc: 50.00%
Val Acc: 50.00% | Duration: 0.082s
------------------------------
Epoch [5/400]
Train Loss: 0.0890 | Train Acc: 75.00%
Val Acc: 50.00% | Duration: 0.098s
------------------------------
Epoch [6/400]
Train Loss: 0.0969 | Train Acc: 50.00%
Val Acc: 50.00% | Duration: 0.067s
------------------------------
Epoch [7/400]
Train Loss: 0.1030 | Train Acc: 25.00%
Val Acc: 50.00% | Duration: 0.071s
------------------------------
Epoch [8/400]
Train Loss: 0.0929 | Train Acc: 50.00%
Val Acc: 50.00% | Duration: 0.071s
------------------------------
Epoch [9/400]
Train Loss: 0.0959 | Train Acc: 50

In [28]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
num_epochs = 50


losses = []
train_accs = []
test_accs = []

for epoch in range(num_epochs):
    model.train()
    start_time = time.perf_counter()
    train_loss = 0.0
    train_correct_count = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss_val = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()
        
        losses.append(loss_val.item())
        train_loss += loss_val.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct_count += (predicted == labels).sum().item()

    train_acc = 100 * train_correct_count / train_total
    val_acc = evaluate_metric(model, val_loader, correct) * 100
    
    train_accs.append(train_acc / 100)
    test_accs.append(val_acc / 100)
    
    end_time = time.perf_counter()
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Acc: {val_acc:.2f}% | Duration: {end_time - start_time:.3f}s")
    print("-" * 30)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Training batch')
plt.ylabel('Cross entropy loss')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Training accuracy')
plt.plot(test_accs, label='Validation accuracy')
plt.xlabel('Epoch')
plt.legend(loc='best')
plt.show()

Epoch [1/50]
Train Loss: 0.7501 | Train Acc: 32.26%
Val Acc: 76.09% | Duration: 2.378s
------------------------------
Epoch [2/50]
Train Loss: 0.6511 | Train Acc: 64.06%
Val Acc: 76.09% | Duration: 2.349s
------------------------------
Epoch [3/50]
Train Loss: 0.6165 | Train Acc: 69.59%
Val Acc: 76.09% | Duration: 2.192s
------------------------------
Epoch [4/50]
Train Loss: 0.6043 | Train Acc: 72.35%
Val Acc: 76.09% | Duration: 2.055s
------------------------------
Epoch [5/50]
Train Loss: 0.6118 | Train Acc: 72.35%
Val Acc: 76.09% | Duration: 2.059s
------------------------------
Epoch [6/50]
Train Loss: 0.5919 | Train Acc: 71.43%
Val Acc: 76.09% | Duration: 2.344s
------------------------------
Epoch [7/50]
Train Loss: 0.5947 | Train Acc: 71.89%
Val Acc: 76.09% | Duration: 2.224s
------------------------------
Epoch [8/50]
Train Loss: 0.5907 | Train Acc: 73.73%
Val Acc: 76.09% | Duration: 2.268s
------------------------------
Epoch [9/50]
Train Loss: 0.5939 | Train Acc: 72.35%
Val 

KeyboardInterrupt: 

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
num_epochs = 10

losses = []
train_accs = []
test_accs = []


best_val_acc = 0.0
patience = 3    
counter = 0    
best_model_path = 'best_model3.pth'


for epoch in range(num_epochs):
    model.train()
    start_time = time.perf_counter()
    train_loss = 0.0
    train_correct_count = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        
        outputs = model(images)
        loss_val = criterion(outputs, labels)
        
        
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()
        
        
        losses.append(loss_val.item())
        train_loss += loss_val.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct_count += (predicted == labels).sum().item()

    
    model.eval() 
    with torch.no_grad():
        val_acc_raw = evaluate_metric(model, val_loader, correct)
        val_acc = val_acc_raw * 100
    
    train_acc = 100 * train_correct_count / train_total
    train_accs.append(train_acc / 100)
    test_accs.append(val_acc_raw)
    
    end_time = time.perf_counter()
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Acc: {val_acc:.2f}% | Duration: {end_time - start_time:.3f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Best model saved (Val Acc: {val_acc:.2f}%)")
    else:
        counter += 1
        print(f"No improvement for {counter} epoch(s).")
    
    if counter >= patience:
        print(f"\nEarly stopping triggered! Training stopped at epoch {epoch+1}")
        break
    print("-" * 30)

model.load_state_dict(torch.load(best_model_path))
print(f"\nLoaded best model with Validation Accuracy: {best_val_acc:.2f}%")

Epoch [1/10]
Train Loss: 0.7242 | Train Acc: 65.90%
Val Acc: 78.26% | Duration: 2.100s
--> Best model saved (Val Acc: 78.26%)
------------------------------
Epoch [2/10]
Train Loss: 0.7103 | Train Acc: 62.67%
Val Acc: 78.26% | Duration: 2.248s
No improvement for 1 epoch(s).
------------------------------
Epoch [3/10]
Train Loss: 0.6644 | Train Acc: 70.97%
Val Acc: 78.26% | Duration: 1.968s
No improvement for 2 epoch(s).
------------------------------
Epoch [4/10]
Train Loss: 0.6867 | Train Acc: 69.12%
Val Acc: 78.26% | Duration: 2.004s
No improvement for 3 epoch(s).

Early stopping triggered! Training stopped at epoch 4

Loaded best model with Validation Accuracy: 78.26%


C:\Users\kiera\AppData\Local\Temp\ipykernel_24220\2002944069.py:71: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))


In [8]:
model.eval()

with torch.no_grad():
    test_acc = evaluate_metric(model, test_loader, correct)

print(f"Final Test Accuracy: {test_acc * 100:.2f}%")

Final Test Accuracy: 77.08%
